<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/02_breakhis_all_magnification_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BreakHis All-Magnification Audit

This notebook extends the completed 40× BreakHis audit to the 100×, 200×, and 400× magnifications.

The purpose is to determine whether the additional magnifications can be used as natural acquisition-shift evaluation conditions while preserving the existing patient-level experimental design.

The existing 40× experiment remains the reference condition. The original patient-level split is not regenerated in this notebook.

This audit addresses the following questions:

1. Which patient IDs appear at 40×, 100×, 200×, and 400×?
2. Are the same patients represented consistently across magnifications?
3. Are diagnosis labels consistent for each patient across magnifications?
4. Are subtype labels consistent across magnifications?
5. Does the previously identified `14-13412` ductal/lobular anomaly appear at the other magnifications?
6. Are there additional duplicate or labeling anomalies outside the 40× subset?
7. Can the existing Development, Calibration, and Test patient assignments be reused for every magnification condition?

No model training or held-out performance evaluation is performed in this notebook.

In [ ]:
# ============================================================
# Environment
# ============================================================

from pathlib import Path
import re
import hashlib

import numpy as np
import pandas as pd
from PIL import Image

SEED = 631

print("All-magnification BreakHis audit")
print("Seed:", SEED)

All-magnification BreakHis audit
Seed: 631


In [ ]:
!pip install -q kagglehub

import kagglehub

breakhis_path = kagglehub.dataset_download(
    "ambarish/breakhis"
)

DATASET_ROOT = Path(breakhis_path)

print("Dataset root:")
print(DATASET_ROOT)

100%|██████████| 3.99G/3.99G [00:43<00:00, 99.5MB/s]

Extracting files...


Dataset root:
/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4


In [ ]:
# ============================================================
# Find all original BreakHis PNG images
# ============================================================

all_pngs = sorted(
    DATASET_ROOT.rglob("*.png")
)

print("Total PNG files:", len(all_pngs))

assert len(all_pngs) == 7909

Total PNG files: 7909


In [ ]:
# ============================================================
# Parse filename metadata
# ============================================================

pattern = re.compile(
    r"^SOB_([BM])_([A-Z]+)-(.+)-(40|100|200|400)-(\d+)\.png$"
)

diagnosis_map = {
    "B": "Benign",
    "M": "Malignant"
}

subtype_map = {
    "A": "Adenosis",
    "F": "Fibroadenoma",
    "TA": "Tubular adenoma",
    "PT": "Phyllodes tumor",
    "DC": "Ductal carcinoma",
    "LC": "Lobular carcinoma",
    "MC": "Mucinous carcinoma",
    "PC": "Papillary carcinoma"
}

rows = []

failed = []

for filepath in all_pngs:

    match = pattern.match(
        filepath.name
    )

    if match is None:
        failed.append(filepath.name)
        continue

    diagnosis_code = match.group(1)
    subtype_code = match.group(2)
    patient_id = match.group(3)
    magnification = match.group(4)
    image_number = int(match.group(5))

    rows.append({
        "filename": filepath.name,
        "filepath": str(filepath),
        "patient_id": patient_id,
        "diagnosis_code": diagnosis_code,
        "diagnosis": diagnosis_map[diagnosis_code],
        "subtype_code": subtype_code,
        "subtype": subtype_map[subtype_code],
        "magnification": magnification,
        "image_number": image_number
    })

metadata_all = pd.DataFrame(rows)

print("Parsed images:", len(metadata_all))
print("Failed filenames:", len(failed))

assert len(metadata_all) == 7909
assert len(failed) == 0

Parsed images: 7909
Failed filenames: 0


In [ ]:
# ============================================================
# Magnification counts
# ============================================================

magnification_counts = (
    metadata_all[
        "magnification"
    ]
    .value_counts()
    .sort_index()
)

display(magnification_counts)

assert magnification_counts["40"] == 1995
assert magnification_counts["100"] == 2081
assert magnification_counts["200"] == 2013
assert magnification_counts["400"] == 1820

,count
magnification,
100,2081
200,2013
40,1995
400,1820


In [ ]:
# ============================================================
# Unique patients by magnification
# ============================================================

patient_counts = (
    metadata_all
    .groupby("magnification")[
        "patient_id"
    ]
    .nunique()
)

display(patient_counts)

,patient_id
magnification,
100,81
200,81
40,81
400,81


In [ ]:
patients_by_mag = {
    mag: set(
        metadata_all.loc[
            metadata_all["magnification"] == mag,
            "patient_id"
        ]
    )
    for mag in ["40", "100", "200", "400"]
}

for mag, patients in patients_by_mag.items():
    print(
        mag,
        "patients:",
        len(patients)
    )

40 patients: 81
100 patients: 81
200 patients: 81
400 patients: 81


In [ ]:
# ============================================================
# Compare patient membership across magnifications
# ============================================================

reference_patients = patients_by_mag["40"]

for mag in ["100", "200", "400"]:

    missing_from_mag = (
        reference_patients -
        patients_by_mag[mag]
    )

    extra_in_mag = (
        patients_by_mag[mag] -
        reference_patients
    )

    print(f"\n40× vs {mag}×")

    print(
        "Missing relative to 40×:",
        len(missing_from_mag)
    )

    print(
        sorted(missing_from_mag)
    )

    print(
        "Extra relative to 40×:",
        len(extra_in_mag)
    )

    print(
        sorted(extra_in_mag)
    )


40× vs 100×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]

40× vs 200×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]

40× vs 400×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]


In [ ]:
# ============================================================
# Patient diagnosis consistency
# ============================================================

patient_diagnosis_counts = (
    metadata_all
    .groupby("patient_id")[
        "diagnosis"
    ]
    .nunique()
)

diagnosis_conflicts = (
    patient_diagnosis_counts[
        patient_diagnosis_counts > 1
    ]
)

print(
    "Patients with conflicting "
    "binary diagnosis labels:",
    len(diagnosis_conflicts)
)

display(diagnosis_conflicts)

Patients with conflicting binary diagnosis labels: 0


,diagnosis
patient_id,


In [ ]:
# ============================================================
# Patient subtype consistency
# ============================================================

patient_subtype_counts = (
    metadata_all
    .groupby("patient_id")[
        "subtype"
    ]
    .nunique()
)

subtype_conflicts = (
    patient_subtype_counts[
        patient_subtype_counts > 1
    ]
)

print(
    "Patients with multiple subtype labels:",
    len(subtype_conflicts)
)

display(subtype_conflicts)

Patients with multiple subtype labels: 1


,subtype
patient_id,
14-13412,2


In [ ]:
# ============================================================
# Patient subtype consistency
# ============================================================

patient_subtype_counts = (
    metadata_all
    .groupby("patient_id")[
        "subtype"
    ]
    .nunique()
)

subtype_conflicts = (
    patient_subtype_counts[
        patient_subtype_counts > 1
    ]
)

print(
    "Patients with multiple subtype labels:",
    len(subtype_conflicts)
)

display(subtype_conflicts)

Patients with multiple subtype labels: 1


,subtype
patient_id,
14-13412,2


In [ ]:
# ============================================================
# Known DC / LC anomaly
# ============================================================

anomaly_patient = (
    metadata_all[
        metadata_all[
            "patient_id"
        ] == "14-13412"
    ]
    .copy()
)

print(
    "Total files for 14-13412:",
    len(anomaly_patient)
)

display(
    anomaly_patient
    .groupby(
        [
            "magnification",
            "subtype"
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

Total files for 14-13412: 246


,magnification,subtype,count
0,100,Ductal carcinoma,33
1,100,Lobular carcinoma,33
2,200,Ductal carcinoma,32
3,200,Lobular carcinoma,32
4,40,Ductal carcinoma,32
5,40,Lobular carcinoma,32
6,400,Ductal carcinoma,26
7,400,Lobular carcinoma,26


In [ ]:
# ============================================================
# Patient × magnification image counts
# ============================================================

patient_mag_counts = (
    metadata_all
    .groupby(
        [
            "patient_id",
            "magnification"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    patient_mag_counts.head(20)
)

print(
    "\nPatients missing at least "
    "one magnification:"
)

missing_any_mag = (
    patient_mag_counts == 0
).any(axis=1)

display(
    patient_mag_counts[
        missing_any_mag
    ]
)

magnification,100,200,40,400
patient_id,,,,
14-10147,12,22,15,15
14-10926,10,9,11,9
14-11031,17,14,14,15
14-11520,26,24,25,23
14-11951,25,28,27,20
14-12204,31,28,20,24
14-12312,41,35,32,38
14-12465,21,19,21,13
14-12773,25,21,26,18



Patients missing at least one magnification:


magnification,100,200,40,400
patient_id,,,,


## All-Magnification Pixel Duplicate Audit

The 40× audit previously identified pixel-identical duplicate images, including a ductal/lobular labeling anomaly for patient `14-13412` and two additional duplicate pairs for patient `14-12204`.

Because 100×, 200×, and 400× are now being considered as magnification-shift evaluation conditions, the duplicate audit is extended to all 7,909 original BreakHis images.

Decoded RGB pixel values are hashed rather than PNG file bytes. This is important because two PNG files may encode identical image pixels while differing at the file-byte level.

The audit distinguishes between:

- duplicates within the same patient and magnification,
- identical images appearing under different magnification labels,
- identical images appearing under different patient IDs.

Duplicate removal is not performed across magnification conditions automatically. Cross-magnification or cross-patient duplicates are treated as potential data-quality anomalies requiring investigation.

In [ ]:
# ============================================================
# Compute decoded-pixel hashes for all 7,909 images
# ============================================================

from tqdm.auto import tqdm

def pixel_hash(filepath):

    with Image.open(filepath) as img:

        arr = np.asarray(
            img.convert("RGB")
        )

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()


pixel_hashes = []

for filepath in tqdm(
    metadata_all["filepath"],
    total=len(metadata_all),
    desc="Hashing BreakHis images"
):

    pixel_hashes.append(
        pixel_hash(filepath)
    )


metadata_all["pixel_hash"] = (
    pixel_hashes
)

print(
    "Images hashed:",
    len(metadata_all)
)

print(
    "Unique pixel hashes:",
    metadata_all[
        "pixel_hash"
    ].nunique()
)

Hashing BreakHis images:   0%|          | 0/7909 [00:00<?, ?it/s]

Images hashed: 7909
Unique pixel hashes: 7784


In [ ]:
# ============================================================
# Identify duplicate pixel-hash groups
# ============================================================

hash_counts = (
    metadata_all[
        "pixel_hash"
    ]
    .value_counts()
)

duplicate_hashes = (
    hash_counts[
        hash_counts > 1
    ]
    .index
)

duplicate_rows = (
    metadata_all[
        metadata_all[
            "pixel_hash"
        ].isin(
            duplicate_hashes
        )
    ]
    .copy()
)

print(
    "Duplicate pixel-hash groups:",
    len(duplicate_hashes)
)

print(
    "Files involved in duplicate groups:",
    len(duplicate_rows)
)

Duplicate pixel-hash groups: 125
Files involved in duplicate groups: 250


In [ ]:
# ============================================================
# Classify duplicate groups
# ============================================================

duplicate_group_summary = []

for pixel_hash_value, group in (
    duplicate_rows
    .groupby("pixel_hash")
):

    duplicate_group_summary.append({

        "pixel_hash":
            pixel_hash_value,

        "n_files":
            len(group),

        "n_patients":
            group[
                "patient_id"
            ].nunique(),

        "n_magnifications":
            group[
                "magnification"
            ].nunique(),

        "n_diagnoses":
            group[
                "diagnosis"
            ].nunique(),

        "n_subtypes":
            group[
                "subtype"
            ].nunique(),

        "patients":
            " | ".join(
                sorted(
                    group[
                        "patient_id"
                    ].unique()
                )
            ),

        "magnifications":
            " | ".join(
                sorted(
                    group[
                        "magnification"
                    ].unique()
                )
            ),

        "subtypes":
            " | ".join(
                sorted(
                    group[
                        "subtype"
                    ].unique()
                )
            )
    })


duplicate_group_summary = pd.DataFrame(
    duplicate_group_summary
)

display(
    duplicate_group_summary.head(20)
)

,pixel_hash,n_files,n_patients,n_magnifications,n_diagnoses,n_subtypes,patients,magnifications,subtypes
0,008398d72570aa0b283eb97c9d4d926863ff889904cfe5...,2,1,1,1,2,14-13412,40,Ductal carcinoma | Lobular carcinoma
1,08ec9e20a8fc539dc06fc827519294ff495b4b95eff171...,2,1,1,1,2,14-13412,100,Ductal carcinoma | Lobular carcinoma
2,09e034ae79e2aad4188072ca84a6ebb7a83ea2744c840d...,2,1,1,1,2,14-13412,200,Ductal carcinoma | Lobular carcinoma
3,0b099edea4d67cfdf9dd2b2a331d34803035e69b434673...,2,1,1,1,2,14-13412,40,Ductal carcinoma | Lobular carcinoma
4,0d4d7781c756115562d98cb5679150fd1ead3a7e43c41c...,2,1,1,1,2,14-13412,400,Ductal carcinoma | Lobular carcinoma
5,0d87e92081cc54e304c97ad2c322fc753510b3974537b7...,2,1,1,1,2,14-13412,40,Ductal carcinoma | Lobular carcinoma
6,0e5d6981765a18830bc6dc2ba5fcffdc701e1ae85f4d05...,2,1,1,1,2,14-13412,40,Ductal carcinoma | Lobular carcinoma
7,0f8b8520a2e72eec615a28c6e65eaf22f3cfc17c43539a...,2,1,1,1,2,14-13412,200,Ductal carcinoma | Lobular carcinoma
8,0ff6ec69a822d4144355cc7816241ffbd8b57eeebbb6a0...,2,1,1,1,2,14-13412,200,Ductal carcinoma | Lobular carcinoma
9,1092bcbe126fb8d1cea41c9c0973372c77697922112529...,2,1,1,1,2,14-13412,200,Ductal carcinoma | Lobular carcinoma


In [ ]:
cross_patient_duplicates = (
    duplicate_group_summary[
        duplicate_group_summary[
            "n_patients"
        ] > 1
    ]
)

print(
    "Duplicate groups spanning "
    "multiple patients:",
    len(cross_patient_duplicates)
)

display(
    cross_patient_duplicates
)

Duplicate groups spanning multiple patients: 0


,pixel_hash,n_files,n_patients,n_magnifications,n_diagnoses,n_subtypes,patients,magnifications,subtypes


In [ ]:
cross_magnification_duplicates = (
    duplicate_group_summary[
        duplicate_group_summary[
            "n_magnifications"
        ] > 1
    ]
)

print(
    "Duplicate groups spanning "
    "multiple magnifications:",
    len(
        cross_magnification_duplicates
    )
)

display(
    cross_magnification_duplicates
)

Duplicate groups spanning multiple magnifications: 0


,pixel_hash,n_files,n_patients,n_magnifications,n_diagnoses,n_subtypes,patients,magnifications,subtypes


In [ ]:
duplicate_rows_by_mag = (
    duplicate_rows
    .groupby(
        "magnification"
    )
    .size()
)

display(
    duplicate_rows_by_mag
)

,0
magnification,
100,66
200,64
40,68
400,52


In [ ]:
same_mag_duplicate_groups = (
    duplicate_rows
    .groupby(
        [
            "magnification",
            "pixel_hash"
        ]
    )
    .size()
    .reset_index(
        name="n_files"
    )
)

same_mag_duplicate_groups = (
    same_mag_duplicate_groups[
        same_mag_duplicate_groups[
            "n_files"
        ] > 1
    ]
)

print(
    "Duplicate groups by magnification:"
)

display(
    same_mag_duplicate_groups
    .groupby(
        "magnification"
    )
    .size()
    .rename(
        "duplicate_groups"
    )
)

Duplicate groups by magnification:


,duplicate_groups
magnification,
100,33
200,32
40,34
400,26


In [ ]:
duplicate_patient_summary = (
    duplicate_rows
    .groupby(
        [
            "magnification",
            "patient_id"
        ]
    )
    .size()
    .reset_index(
        name="files_in_duplicate_groups"
    )
)

display(
    duplicate_patient_summary
    .sort_values(
        [
            "magnification",
            "files_in_duplicate_groups"
        ],
        ascending=[
            True,
            False
        ]
    )
)

,magnification,patient_id,files_in_duplicate_groups
0,100,14-13412,66
1,200,14-13412,64
3,40,14-13412,64
2,40,14-12204,4
4,400,14-13412,52


In [ ]:
anomaly_hash_audit = (
    duplicate_rows[
        duplicate_rows[
            "patient_id"
        ] == "14-13412"
    ]
    .sort_values(
        [
            "magnification",
            "image_number",
            "subtype"
        ]
    )
)

display(
    anomaly_hash_audit[
        [
            "magnification",
            "image_number",
            "subtype",
            "filename",
            "pixel_hash"
        ]
    ]
)

,magnification,image_number,subtype,filename,pixel_hash
2923,100,1,Ductal carcinoma,SOB_M_DC-14-13412-100-001.png,238c30f90acd0d3b3bdf2464c07e54161af459009bc0f2...
6034,100,1,Lobular carcinoma,SOB_M_LC-14-13412-100-001.png,238c30f90acd0d3b3bdf2464c07e54161af459009bc0f2...
2924,100,2,Ductal carcinoma,SOB_M_DC-14-13412-100-002.png,47b8ba25e3eb0e42e572c33d43c2106ed34e688fe1deb1...
6035,100,2,Lobular carcinoma,SOB_M_LC-14-13412-100-002.png,47b8ba25e3eb0e42e572c33d43c2106ed34e688fe1deb1...
2925,100,3,Ductal carcinoma,SOB_M_DC-14-13412-100-003.png,a665748cc817dafee99a8965f447ace24ef3e5a6e82ad6...
...,...,...,...,...,...
3011,400,25,Ductal carcinoma,SOB_M_DC-14-13412-400-025.png,2cd8f803f186bbc954540d7601f33cca45c124d5abfcfb...
6123,400,25,Lobular carcinoma,SOB_M_LC-14-13412-400-025.png,4fec473461ce5e6b1387963ab6fb3546b457979e1aa8ad...
3012,400,26,Ductal carcinoma,SOB_M_DC-14-13412-400-026.png,4fec473461ce5e6b1387963ab6fb3546b457979e1aa8ad...
6124,400,26,Lobular carcinoma,SOB_M_LC-14-13412-400-026.png,7e0005e5ea5f0388fc0c02b5d34094d79fb2523953726b...


In [ ]:
anomaly_duplicate_summary = (
    anomaly_hash_audit
    .groupby(
        [
            "magnification",
            "pixel_hash"
        ]
    )
    .agg(
        n_files=(
            "filename",
            "size"
        ),

        n_subtypes=(
            "subtype",
            "nunique"
        ),

        subtypes=(
            "subtype",
            lambda x:
                " | ".join(
                    sorted(set(x))
                )
        )
    )
    .reset_index()
)

display(
    anomaly_duplicate_summary
    .groupby(
        "magnification"
    )
    .agg(
        duplicate_groups=(
            "pixel_hash",
            "size"
        ),

        groups_spanning_subtypes=(
            "n_subtypes",
            lambda x:
                int((x > 1).sum())
        )
    )
)

,duplicate_groups,groups_spanning_subtypes
magnification,,
100,33,33
200,32,32
40,32,32
400,26,26


In [ ]:
patient_12204_duplicates = (
    duplicate_rows[
        duplicate_rows[
            "patient_id"
        ] == "14-12204"
    ]
    .sort_values(
        [
            "magnification",
            "image_number"
        ]
    )
)

display(
    patient_12204_duplicates[
        [
            "magnification",
            "image_number",
            "subtype",
            "filename",
            "pixel_hash"
        ]
    ]
)

,magnification,image_number,subtype,filename,pixel_hash
6029,40,16,Lobular carcinoma,SOB_M_LC-14-12204-40-016.png,c5bea6c04c60d094c75a31029a48734712260e3cef0325...
6030,40,17,Lobular carcinoma,SOB_M_LC-14-12204-40-017.png,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...
6032,40,34,Lobular carcinoma,SOB_M_LC-14-12204-40-034.png,c5bea6c04c60d094c75a31029a48734712260e3cef0325...
6033,40,35,Lobular carcinoma,SOB_M_LC-14-12204-40-035.png,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...


In [ ]:
# ============================================================
# Check DC/LC duplicate-pair alignment for patient 14-13412
# ============================================================

pair_alignment_rows = []

patient_13412_duplicates = (
    duplicate_rows[
        duplicate_rows["patient_id"] == "14-13412"
    ]
    .copy()
)

for (mag, pixel_hash_value), group in (
    patient_13412_duplicates
    .groupby(
        ["magnification", "pixel_hash"]
    )
):

    image_numbers = sorted(
        group["image_number"].tolist()
    )

    filenames = sorted(
        group["filename"].tolist()
    )

    subtypes = sorted(
        group["subtype"].tolist()
    )

    pair_alignment_rows.append({
        "magnification": mag,
        "pixel_hash": pixel_hash_value,
        "n_files": len(group),
        "image_numbers":
            " | ".join(
                map(str, image_numbers)
            ),
        "same_image_number":
            len(set(image_numbers)) == 1,
        "subtypes":
            " | ".join(subtypes),
        "filenames":
            " | ".join(filenames)
    })


pair_alignment = pd.DataFrame(
    pair_alignment_rows
)

alignment_summary = (
    pair_alignment
    .groupby("magnification")
    .agg(
        duplicate_pairs=(
            "pixel_hash",
            "size"
        ),
        same_number_pairs=(
            "same_image_number",
            "sum"
        )
    )
)

alignment_summary[
    "different_number_pairs"
] = (
    alignment_summary[
        "duplicate_pairs"
    ]
    -
    alignment_summary[
        "same_number_pairs"
    ]
)

display(alignment_summary)

,duplicate_pairs,same_number_pairs,different_number_pairs
magnification,,,
100,33,33,0
200,32,10,22
40,32,32,0
400,26,14,12


In [ ]:
display(
    pair_alignment[
        ~pair_alignment[
            "same_image_number"
        ]
    ][
        [
            "magnification",
            "image_numbers",
            "subtypes",
            "filenames"
        ]
    ]
)

,magnification,image_numbers,subtypes,filenames
35,200,18 | 19,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-019.png | SOB_M_LC-14-13...
36,200,25 | 26,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-026.png | SOB_M_LC-14-13...
37,200,11 | 12,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-012.png | SOB_M_LC-14-13...
39,200,12 | 13,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-013.png | SOB_M_LC-14-13...
40,200,20 | 21,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-021.png | SOB_M_LC-14-13...
41,200,27 | 28,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-028.png | SOB_M_LC-14-13...
42,200,17 | 18,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-018.png | SOB_M_LC-14-13...
43,200,31 | 32,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-032.png | SOB_M_LC-14-13...
44,200,13 | 14,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-014.png | SOB_M_LC-14-13...
45,200,19 | 20,Ductal carcinoma | Lobular carcinoma,SOB_M_DC-14-13412-200-020.png | SOB_M_LC-14-13...


### Duplicate-pair alignment finding

The pixel-hash audit confirmed that the duplicate-label anomaly for patient `14-13412` occurs at all four magnifications. However, matching duplicate files cannot always be identified using the image number encoded in the filename.

At 40× and 100×, all pixel-identical ductal/lobular duplicate pairs shared the same image number. At 200×, 22 of the 32 duplicate pairs had different image numbers, and at 400×, 12 of the 26 pairs had different image numbers. Many of these mismatches differed by one image index.

Therefore, filename or image-number matching is not used for deduplication. Deduplication is based on SHA-256 hashes of the decoded RGB pixel arrays. Within each patient and magnification, one deterministic copy is retained for each unique pixel hash.

The raw BreakHis files are not modified or deleted. Deduplication is performed only in the analysis metadata.

In [ ]:
# ============================================================
# Prepare provenance and analysis subtype fields
# ============================================================

all_mag_metadata = metadata_all.copy()

all_mag_metadata["original_subtype_code"] = (
    all_mag_metadata["subtype_code"]
)

all_mag_metadata["original_subtype"] = (
    all_mag_metadata["subtype"]
)

all_mag_metadata["analysis_subtype_code"] = (
    all_mag_metadata["original_subtype_code"]
)

all_mag_metadata["analysis_subtype"] = (
    all_mag_metadata["original_subtype"]
)

# Known source-label ambiguity
ambiguous_mask = (
    all_mag_metadata["patient_id"] == "14-13412"
)

all_mag_metadata.loc[
    ambiguous_mask,
    "analysis_subtype_code"
] = "DC/LC"

all_mag_metadata.loc[
    ambiguous_mask,
    "analysis_subtype"
] = "DC/LC ambiguous"

In [ ]:
all_mag_metadata["relative_path"] = (
    all_mag_metadata["filepath"]
    .apply(
        lambda x: str(
            Path(x).relative_to(DATASET_ROOT)
        )
    )
)

In [ ]:
# ============================================================
# Deterministic all-magnification deduplication
# ============================================================

all_mag_metadata = (
    all_mag_metadata
    .sort_values(
        [
            "patient_id",
            "magnification",
            "pixel_hash",
            "filename"
        ]
    )
    .reset_index(drop=True)
)

all_mag_metadata["duplicate_group_size"] = (
    all_mag_metadata
    .groupby(
        [
            "patient_id",
            "magnification",
            "pixel_hash"
        ]
    )["filename"]
    .transform("size")
)

all_mag_metadata["duplicate_rank"] = (
    all_mag_metadata
    .groupby(
        [
            "patient_id",
            "magnification",
            "pixel_hash"
        ]
    )
    .cumcount()
)

all_mag_metadata["excluded_duplicate"] = (
    all_mag_metadata["duplicate_rank"] > 0
)

print(
    "Rows marked as duplicate copies:",
    all_mag_metadata["excluded_duplicate"].sum()
)

assert (
    all_mag_metadata["excluded_duplicate"].sum()
    == 125
)

Rows marked as duplicate copies: 125


In [ ]:
clean_all_mag = (
    all_mag_metadata[
        ~all_mag_metadata["excluded_duplicate"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Raw images:", len(all_mag_metadata))
print(
    "Excluded duplicates:",
    all_mag_metadata["excluded_duplicate"].sum()
)
print("Clean images:", len(clean_all_mag))

assert len(clean_all_mag) == 7784

Raw images: 7909
Excluded duplicates: 125
Clean images: 7784


In [ ]:
clean_counts = (
    clean_all_mag["magnification"]
    .value_counts()
    .sort_index()
)

display(clean_counts)

assert clean_counts["40"] == 1961
assert clean_counts["100"] == 2048
assert clean_counts["200"] == 1981
assert clean_counts["400"] == 1794

,count
magnification,
100,2048
200,1981
40,1961
400,1794


In [ ]:
clean_patient_counts = (
    clean_all_mag
    .groupby("magnification")["patient_id"]
    .nunique()
)

display(clean_patient_counts)

assert (clean_patient_counts == 81).all()

,patient_id
magnification,
100,81
200,81
40,81
400,81


In [ ]:
# ============================================================
# Save all-magnification pixel-hash audit
# ============================================================

from pathlib import Path

METADATA_DIR = Path(
    "/content/breast-histology-calibration/metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Make sure we save portable relative paths,
# not temporary Colab absolute paths.
if "relative_path" not in metadata_all.columns:

    metadata_all["relative_path"] = (
        metadata_all["filepath"]
        .apply(
            lambda x: str(
                Path(x).relative_to(
                    DATASET_ROOT
                )
            )
        )
    )

hash_audit_columns = [
    "filename",
    "relative_path",
    "patient_id",
    "diagnosis_code",
    "diagnosis",
    "subtype_code",
    "subtype",
    "magnification",
    "image_number",
    "pixel_hash"
]

hash_audit_path = (
    METADATA_DIR /
    "breakhis_all_magnification_hash_audit.csv"
)

metadata_all[
    hash_audit_columns
].to_csv(
    hash_audit_path,
    index=False
)

print("Saved:")
print(hash_audit_path)

print("\nRows:", len(metadata_all))

print(
    "Unique hashes:",
    metadata_all["pixel_hash"].nunique()
)

print(
    "File exists:",
    hash_audit_path.exists()
)

print(
    "File size MB:",
    round(
        hash_audit_path.stat().st_size
        / (1024 ** 2),
        2
    )
)

Saved:
/content/breast-histology-calibration/metadata/breakhis_all_magnification_hash_audit.csv

Rows: 7909
Unique hashes: 7784
File exists: True
File size MB: 2.04


## Verify Consistency With the Previously Frozen 40× Dataset

The all-magnification audit introduces a new cleaning procedure that is applied consistently across 40×, 100×, 200×, and 400× BreakHis images. Before using the expanded metadata, the new 40× subset must be compared against the previously frozen 40× metadata created during the original experiment.

The goal of this check is to verify that extending the duplicate audit to all four magnifications has **not changed the already-established 40× analysis cohort**.

The previously frozen file:

`breakhis_40x_final_metadata.csv`

contains the original clean 40× dataset used for the existing Development, Calibration, and Test design.

Because this notebook is running in a fresh Colab environment, the frozen 40× metadata file is loaded separately before comparison.

The consistency check verifies that:

- both versions contain exactly 1,961 clean 40× images,
- the retained filenames are identical,
- patient IDs match,
- binary diagnosis labels match.

If the two 40× datasets differ, the expanded all-magnification metadata must not replace the original frozen dataset until the discrepancy is investigated.

If they match exactly, the new all-magnification deduplication procedure is considered consistent with the previously frozen 40× experiment.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving breakhis_40x_final_metadata.csv to breakhis_40x_final_metadata.csv


In [ ]:
from pathlib import Path
import shutil

METADATA_DIR = Path(
    "/content/breast-histology-calibration/metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

source = Path(
    "/content/breakhis_40x_final_metadata.csv"
)

destination = (
    METADATA_DIR /
    "breakhis_40x_final_metadata.csv"
)

shutil.move(
    source,
    destination
)

print("Exists:", destination.exists())
print("Path:", destination)

Exists: True
Path: /content/breast-histology-calibration/metadata/breakhis_40x_final_metadata.csv


In [ ]:
import pandas as pd

existing_40x = pd.read_csv(
    destination
)

print("Rows:", len(existing_40x))
print(
    "Patients:",
    existing_40x["patient_id"].nunique()
)

display(existing_40x.head())

Rows: 1961
Patients: 81


,filename,relative_path,patient_id,diagnosis_code,diagnosis,original_subtype_code,original_subtype,subtype_code,subtype,magnification,image_number,split
0,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-10147,M,Malignant,MC,Mucinous carcinoma,MC,Mucinous carcinoma,40,9,development
1,SOB_M_MC-14-10147-40-003.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-10147,M,Malignant,MC,Mucinous carcinoma,MC,Mucinous carcinoma,40,3,development
2,SOB_M_MC-14-10147-40-004.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-10147,M,Malignant,MC,Mucinous carcinoma,MC,Mucinous carcinoma,40,4,development
3,SOB_M_MC-14-10147-40-005.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-10147,M,Malignant,MC,Mucinous carcinoma,MC,Mucinous carcinoma,40,5,development
4,SOB_M_MC-14-10147-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-10147,M,Malignant,MC,Mucinous carcinoma,MC,Mucinous carcinoma,40,12,development


In [ ]:
# ============================================================
# Verify new 40× clean subset against existing frozen 40× metadata
# ============================================================

new_40x = (
    clean_all_mag[
        clean_all_mag["magnification"].astype(str) == "40"
    ]
    .copy()
)

old_filenames = set(
    existing_40x["filename"]
)

new_filenames = set(
    new_40x["filename"]
)

print("Existing frozen 40× images:", len(old_filenames))
print("New audit 40× images:", len(new_filenames))

print(
    "Only in existing frozen metadata:",
    len(old_filenames - new_filenames)
)

print(
    "Only in new all-magnification audit:",
    len(new_filenames - old_filenames)
)

Existing frozen 40× images: 1961
New audit 40× images: 1961
Only in existing frozen metadata: 0
Only in new all-magnification audit: 0


In [ ]:
# ============================================================
# Strict consistency assertions
# ============================================================

assert len(old_filenames) == 1961
assert len(new_filenames) == 1961

assert old_filenames == new_filenames

print(
    "PASS: The new all-magnification deduplication "
    "reproduces the exact frozen 40× image set."
)

PASS: The new all-magnification deduplication reproduces the exact frozen 40× image set.


In [ ]:
# ============================================================
# Verify patient and diagnosis consistency at 40×
# ============================================================

old_check = (
    existing_40x[
        [
            "filename",
            "patient_id",
            "diagnosis"
        ]
    ]
    .sort_values("filename")
    .reset_index(drop=True)
)

new_check = (
    new_40x[
        [
            "filename",
            "patient_id",
            "diagnosis"
        ]
    ]
    .sort_values("filename")
    .reset_index(drop=True)
)

comparison = old_check.merge(
    new_check,
    on="filename",
    how="outer",
    suffixes=(
        "_existing",
        "_new"
    ),
    indicator=True
)

patient_mismatches = (
    comparison[
        comparison["patient_id_existing"]
        != comparison["patient_id_new"]
    ]
)

diagnosis_mismatches = (
    comparison[
        comparison["diagnosis_existing"]
        != comparison["diagnosis_new"]
    ]
)

print(
    "Patient ID mismatches:",
    len(patient_mismatches)
)

print(
    "Diagnosis mismatches:",
    len(diagnosis_mismatches)
)

assert len(patient_mismatches) == 0
assert len(diagnosis_mismatches) == 0

print(
    "PASS: Patient IDs and binary diagnoses "
    "also match the frozen 40× metadata."
)

Patient ID mismatches: 0
Diagnosis mismatches: 0
PASS: Patient IDs and binary diagnoses also match the frozen 40× metadata.


In [ ]:
# ============================================================
# Save clean all-magnification metadata
# ============================================================

clean_output_columns = [
    "filename",
    "relative_path",
    "patient_id",
    "diagnosis_code",
    "diagnosis",
    "original_subtype_code",
    "original_subtype",
    "analysis_subtype_code",
    "analysis_subtype",
    "magnification",
    "image_number",
    "pixel_hash",
    "duplicate_group_size",
    "duplicate_rank",
    "excluded_duplicate"
]

clean_output_path = (
    METADATA_DIR /
    "breakhis_all_magnification_clean_metadata.csv"
)

clean_all_mag[
    clean_output_columns
].to_csv(
    clean_output_path,
    index=False
)

print("Saved:")
print(clean_output_path)

print(
    "Rows:",
    len(clean_all_mag)
)

print(
    "Patients:",
    clean_all_mag["patient_id"].nunique()
)

print(
    "File exists:",
    clean_output_path.exists()
)

Saved:
/content/breast-histology-calibration/metadata/breakhis_all_magnification_clean_metadata.csv
Rows: 7784
Patients: 81
File exists: True


In [ ]:
summary = (
    clean_all_mag
    .groupby("magnification")
    .agg(
        images=(
            "filename",
            "size"
        ),
        patients=(
            "patient_id",
            "nunique"
        )
    )
)

display(summary)

,images,patients
magnification,,
100,2048,81
200,1981,81
40,1961,81
400,1794,81
